# Introduction to GPU Performance Profiling
---

This notebook introduces the concepts and workflow behind GPU performance profiling for deep learning.  By the end of this short module you will understand *why* we profile, *what* the profiling loop looks like, and *which tools* we will use in the notebooks that follow.

## Why Profile?

GPUs are fast — but they are only fast when kept busy.  A training loop that looks reasonable in code can spend the majority of its time waiting: waiting for data to load, waiting for a CPU→GPU copy to finish, waiting for the CPU to catch up to the GPU.  The gap between what your hardware *could* do and what it *is* doing is where profiling lives.

The goal is not to make your code run in the minimum conceivable time — it is to make sure your hardware is doing the work you think it is doing, and to find the gaps when it is not.

## The Profiling Loop

Performance work always follows the same cycle:

```
  ┌─────────────────────────────────────────────────┐
  │                                                 │
  │   1. Measure  ──→  Correct & fast enough?       │
  │                          │           │          │
  │                         Yes          No         │
  │                          │           │          │
  │                        Stop    2. Profile       │
  │                                    │            │
  │                          3. Interpret results   │
  │                                    │            │
  │                          4. Form a hypothesis   │
  │                                    │            │
  │                          5. Make one change     │
  │                                    │            │
  └────────────────────────────────────┘            │
                                                    │
  (loop back to step 1)  ◄──────────────────────────┘
```

**Step 1 — Measure** (timing + correctness check)  
Always start with a wall-clock timer.  Are the results numerically correct?  Is the step time reasonable for the hardware?  If yes, you are done.  If no, reach for a profiler.

**Step 2 — Profile**  
Run the code under a profiler that records *where time actually goes*, not where you expect it to go.  Human intuition about GPU bottlenecks is almost always wrong.

**Step 3 — Interpret results**  
Read the profiler output.  Which section of the training loop is slow?  By how much?

**Step 4 — Form a hypothesis**  
*"The DataLoader section takes 70 % of each step.  I think the bottleneck is single-threaded data loading."*  One hypothesis at a time.

**Step 5 — Make one change**  
Change exactly one thing.  Go back to step 1.  If the timing improved, keep it.  If not, revert.

Changing multiple things at once makes it impossible to know what helped.

## A Quick Timing Demo

Before reaching for any profiling tool, always start with a simple wall-clock timer.  In Python the right tool is `time.perf_counter()` — it has nanosecond resolution and is not affected by system clock adjustments.

The cell below runs [`train_v1.py`](../source_code/intro/train_v1.py), a minimal CIFAR-10 + ResNet18 training loop, and prints the mean step time and throughput.  No profiling yet — just a baseline measurement.

In [ ]:
!python ../source_code/intro/train_v1.py

Take note of the step time and throughput.  We will use these numbers as our starting point throughout the next two notebooks.

## The Two Tools We Will Use

Different profiling questions need different tools.  Across the next few notebooks we will use two that sit at different levels of the stack.

### PyTorch Profiler

Answers the question: **which section of my training step is slow?**

`torch.profiler.profile()` is a context manager you wrap around your training loop.  It records the wall-clock time spent in each labelled section (DataLoader, forward, backward, optimizer) and writes a trace that can be viewed in TensorBoard.  It is fast to set up, the output is immediately readable, and it covers 80 % of common bottlenecks.

What it *cannot* tell you: *why* a section is slow.  It sees sections, not the GPU timeline.

### NVIDIA Nsight Systems (`nsys`)

Answers the question: **what is the GPU actually doing, and when?**

`nsys profile` wraps your training script and records a complete timeline of CPU and GPU activity — every CUDA kernel, every host↔device memory copy, every synchronisation point.  The resulting `.nsys-rep` file can be opened in the Nsight Systems GUI where you can zoom into individual steps and see exactly what is happening at microsecond resolution.

Nsight Systems makes *concurrency* and *synchronisation* problems visible — things that are completely invisible in a step-level view.

### The Workflow

```
  Slow step?
      │
      ▼
  torch.profiler  →  "backward section is slow"
      │                       │
      │               Not obvious why?
      │                       │
      ▼                       ▼
   Fixed             nsys  →  GPU timeline  →  root cause
```

## What's Ahead

| Notebook | Tool | Bug | Fix |
|---|---|---|---|
| **PyTorch Profiler** | `torch.profiler` | DataLoader stall | `num_workers`, `pin_memory` |
| **AMP & Profiler Limits** | `torch.profiler` | Slow backward (reason unclear) | Investigate further |
| **Nsight Systems** | `nsys` | Per-op sync in backward | Remove debug flag |

Each notebook follows the same loop: measure → profile → hypothesise → fix → measure again.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — PyTorch Profiler](intro-pytorch-profiler.ipynb)</b></div></center>

---

## Links and Resources

- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)
- [NVIDIA Nsight Systems](https://developer.nvidia.com/nsight-systems)
- [PyTorch Profiler with TensorBoard tutorial](https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).